# probe_k3 — BẢN SỬA VÙNG ĐO. `K_CHUNK = 3`, lấy MAX.

**~2,3 giờ GPU** (81.000 cặp: train 36.000 + dev 45.000). Rủi ro hỏng model: **bằng 0**.

### Vì sao phải có bản này

Hai lượt trước chấm mỗi văn bản bằng **1 đoạn** → MỐC 0,8267 (EXC=900) rồi 0,8617 (EXC=1800),
**cả hai đều thua SÀN 0,9117** (= lấy thẳng top-5 của rổ, không chấm gì). Đo Δ trong vùng đó
không đọc được. Tầng 1 thật chấm **3 đoạn rồi lấy MAX** — bản này làm đúng như vậy.

### Ba thứ đã sửa so với `probe_run.ipynb`

1. `K_CHUNK = 3` + `ba_doan()` đệm cho đủ 3 (đệm trùng vô hại vì lấy MAX).
2. Cache đổi tên `vec_*_k3.npy` — **không bao giờ dùng lại tên cũ**, vì `nap()` chỉ kiểm số
   hàng chứ không kiểm nội dung, và vector k=1/k=3 có số hàng khác nhau nhưng vector 900/1800
   thì bằng nhau. Đây là bẫy đã suýt dính.
3. **Lưu `scores_dev300_moc_k3.json`** — điểm của đầu GỐC. Hai lượt trước vứt mất
   (`MOC, _ = cham(...)`), đúng lỗi quy tắc 2, nên không phân tích được từng câu trên CPU.

### Đọc kết quả

`meta_probe.json` có `hop_le`. **`hop_le = false` thì bỏ `dat` đi, đừng đọc.**


In [ ]:
# ===== Bước 0: cấu hình =====
import os, sys, json, glob, time, random
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

MODEL   = "AITeamVN/Vietnamese_Reranker"
N_CAU, K = 1500, 8          # 1 gold + 7 negative BM25 hạng 3-10
K_CHUNK = 3                 # 3 đoạn/văn bản rồi lấy MAX — ĐÚNG tầng 1 thật.
                            # Bản k=1 có MỐC 0.8617 < SÀN 0.9117: một đoạn không đủ.
BO_TOP  = 2                 # bỏ 2 negative dễ-là-gold nhất
EXC, MAXLEN = 1800, 512    # = MERGE_CHARS: KHONG cat. Ban 900 cat doi doan
                           # -> tang 1 tut xuong 0.8267, THUA ro tran 0.9117 (do lai 27/08)
EPOCH, LR, BS = 30, 1e-3, 64
NGUONG  = 0.0100           # doc CUNG dong san o Buoc 4, khong doc mot minh
SEED = 20260825
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "CẦN GPU"

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

def tim(ten):
    p = glob.glob(f"{INPUT_DIR}/**/{ten}", recursive=True)
    assert p, f"KHÔNG THẤY {ten} — upload rồi chạy lại"
    return sorted(p, key=len)[0]

import deep_chunk as DC
from rerank_from_d import blend_bm25_first
DC.MERGE_CHARS = 1800
print("GPU:", torch.cuda.get_device_name(0))

def ba_doan(q, d):
    """K_CHUNK đoạn của văn bản d. Văn bản ít đoạn hơn thì đệm bằng đoạn cuối —
    đệm trùng lặp VÔ HẠI vì ta lấy MAX, max của hai bản sao bằng chính nó."""
    cs = [c[:EXC] for c in DC.pick_chunks(q, CTX_DIR, d, k=K_CHUNK)] or [""]
    return (cs + [cs[-1]] * K_CHUNK)[:K_CHUNK]


In [ ]:
# ===== Bước 1: dựng nhóm huấn luyện =====
train = json.load(open(tim("train.json"), encoding="utf-8"))
bm25  = json.load(open(tim("bm25_ids_train_FALLBACK.json"), encoding="utf-8"))
co_ctx = lambda d: os.path.isfile(f"{CTX_DIR}/context_{d}.json")

qs_all = sorted(q for q in train if q in bm25); random.shuffle(qs_all)
nhom = {}
for q in qs_all:
    if len(nhom) >= N_CAU: break
    gold = [str(a) for a in train[q]["answer"] if co_ctx(str(a))]
    if not gold: continue
    neg = [str(c["doc_id"]) for c in bm25[q]
           if str(c["doc_id"]) not in gold and co_ctx(str(c["doc_id"]))]
    neg = neg[BO_TOP:]                       # bỏ top-2: hay là gold trá hình
    if len(neg) < K - 1: continue
    nhom[q] = gold[:1] + neg[:K-1]           # phần tử 0 LUÔN là gold
print(f"{len(nhom)} nhóm × {K} = {len(nhom)*K:,} cặp huấn luyện")

t0 = time.time(); Xtr, Itr = [], []
for i, (q, docs) in enumerate(nhom.items(), 1):
    qq = train[q]["question"]
    for d in docs:
        for c in ba_doan(qq, d): Xtr.append([qq, c]); Itr.append((q, d))
    if i % 500 == 0: print(f"  băm {i}/{len(nhom)} | {(time.time()-t0)/60:.1f} phút", flush=True)
print(f"{len(Xtr):,} cặp · {(time.time()-t0)/60:.1f} phút")


In [ ]:
# ===== Bước 2: dựng tập dev300 (dùng chung cho mốc và bản probe) =====
dev  = json.load(open(tim("dev_300_locked.json"), encoding="utf-8"))
cand = json.load(open(tim("fusion_rrf_top50_dev_1000_matchEmbedded.json"), encoding="utf-8"))
qd   = [q for q in dev if q in cand]
GOLD = {q: {str(a) for a in dev[q]["answer"]} for q in qd}
ORDER= {q: [str(c["doc_id"]) for c in sorted(cand[q], key=lambda c: -float(c["rrf_score"]))]
        for q in qd}
t0 = time.time(); Xdv, Idv = [], []
for i, q in enumerate(qd, 1):
    qq = dev[q]["question"]
    for d in ORDER[q]:
        for c in ba_doan(qq, d): Xdv.append([qq, c]); Idv.append((q, d))
    if i % 100 == 0: print(f"  băm dev {i}/{len(qd)} | {(time.time()-t0)/60:.1f} phút", flush=True)
print(f"{len(Xdv):,} cặp dev · {(time.time()-t0)/60:.1f} phút")


In [ ]:
# ===== Bước 3: MÃ HOÁ MỘT LẦN, cache vector. Encoder đóng băng từ đây =====
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(MODEL)
mod = AutoModelForSequenceClassification.from_pretrained(MODEL).to("cuda").eval()
n = sum(p.numel() for p in mod.parameters())
print(f"{MODEL}: {n:,} ({n/1e9:.3f}B)"); assert n <= 3_000_000_000

@torch.no_grad()
def ma_hoa(X, bs=64, ten=""):
    V = []
    for i in range(0, len(X), bs):
        b = X[i:i+bs]
        enc = tok([x[0] for x in b], [x[1] for x in b], padding=True, truncation=True,
                  max_length=MAXLEN, return_tensors="pt").to("cuda")
        h = mod.roberta(**enc).last_hidden_state[:, 0]      # vector CLS, 1024 chiều
        V.append(h.float().cpu())
        if i % (bs*50) == 0: print(f"  {ten} {i:,}/{len(X):,}", flush=True)
    return torch.cat(V)

# Nạp lại vector nếu lượt trước đã lưu (upload outputs/ lên dataset).
# Mã hoá mất 28 phút; một lỗi ở bước sau mà phải mã hoá lại là quá đắt.
import numpy as np
def nap(ten, n):
    for p in glob.glob(f"{INPUT_DIR}/**/{ten}", recursive=True) + [f"{OUT}/{ten}"]:
        if os.path.isfile(p):
            v = torch.from_numpy(np.load(p)).float()
            if len(v) == n: print(f"  nạp lại {ten} {tuple(v.shape)} — bỏ qua mã hoá"); return v
    return None

t0 = time.time()
Vtr = nap("vec_train_k3.npy", len(Xtr))
if Vtr is None:
    Vtr = ma_hoa(Xtr, ten="train"); np.save(f"{OUT}/vec_train_k3.npy", Vtr.half().numpy())
Vdv = nap("vec_dev_k3.npy", len(Xdv))
if Vdv is None:
    Vdv = ma_hoa(Xdv, ten="dev");   np.save(f"{OUT}/vec_dev_k3.npy", Vdv.half().numpy())
print(f"\nmã hoá xong · {(time.time()-t0)/60:.1f} phút · train {tuple(Vtr.shape)} · dev {tuple(Vdv.shape)}")



In [ ]:
# ===== Bước 4: mốc — đầu phân loại GỐC, chạy trên chính vector vừa cache =====
import copy
# XLMRobertaClassificationHead tự làm `features[:, 0, :]` -> nó đòi tensor [B, L, H].
# Ta đã bóc sẵn vector CLS thành [N, H], nên phải chèn lại trục độ dài: [N, 1, H].
# BẪY 25/08: đưa thẳng [N, H] -> "too many indices for tensor of dimension 2",
# và chỉ nổ SAU 28 phút mã hoá.
def cham_vec(head, V):
    return head(V.unsqueeze(1)).squeeze(-1)

def cham(head, V, bs=4096):
    head.eval(); s = []
    with torch.no_grad():
        for i in range(0, len(V), bs):
            s += cham_vec(head, V[i:i+bs].to("cuda")).float().cpu().tolist()
    per = {}
    for (q, d), v in zip(Idv, s):          # K_CHUNK hàng liên tiếp cùng một (q, d)
        p = per.setdefault(q, {})
        if v > p.get(d, -1e30): p[d] = v   # MAX — y hệt tầng 1 thật gộp top_chunks
    r = {1: 0.0, 5: 0.0}
    for q in qd:
        p = blend_bm25_first(sorted(per[q], key=lambda d: -per[q][d]), ORDER[q], k=5, n_bm25=1)
        for k in r: r[k] += len(GOLD[q] & set(p[:k])) / len(GOLD[q])
    return {k: v/len(qd) for k, v in r.items()}, per

head0 = copy.deepcopy(mod.classifier).to("cuda")      # KHỞI TẠO BẰNG TRỌNG SỐ GỐC
MOC, PER_MOC = cham(head0, Vdv)   # GIỮ điểm: quy tắc 2, lượt trước vứt mất

# SÀN = lấy thẳng top-5 của rổ, KHÔNG chấm lại gì. Mọi bộ chấm phải hơn dòng này.
# 27/08: bản EXC=900 có MỐC 0.8267 < SÀN 0.9117 -> đo Δ trong vùng hỏng, cổng vô nghĩa.
SAN = sum(len(GOLD[q] & set(blend_bm25_first([], ORDER[q], k=5, n_bm25=1))) / len(GOLD[q])
          for q in qd) / len(qd)
print(f"SÀN (rổ trần, không reranker)  R@5={SAN:.4f}")
print(f"MỐC (đầu gốc)  R@1={MOC[1]:.4f}  R@5={MOC[5]:.4f}")
HOP_LE = MOC[5] >= SAN
if not HOP_LE:
    print("\n" + "!"*70)
    print(f"CẢNH BÁO: MỐC {MOC[5]:.4f} < SÀN {SAN:.4f} — vùng đo VẪN chưa hợp lệ.")
    print("Vector đã lưu rồi nên chạy tiếp chỉ tốn ~2 phút; Δ vẫn đáng ghi, NHƯNG")
    print("KHÔNG được đọc nó như một cổng. Đừng chạy LoRA dựa trên nó.")
    print("!"*70 + "\n")



In [ ]:
# ===== Bước 5: train ĐẦU PHÂN LOẠI. Encoder không đụng tới một tham số nào =====
head = copy.deepcopy(mod.classifier).to("cuda")       # bắt đầu ĐÚNG từ mốc
for p in mod.parameters(): p.requires_grad = False    # nói rõ ý định
opt = torch.optim.AdamW(head.parameters(), lr=LR)
Vtr_g = Vtr.view(len(nhom), K, K_CHUNK, -1).to("cuda") # [nhóm, 8, 3, 1024]
tgt = torch.zeros(len(nhom), dtype=torch.long, device="cuda")   # gold LUÔN ở chỉ số 0
idx = list(range(len(nhom)))
print(f"tham số học: {sum(p.numel() for p in head.parameters()):,} "
      f"({sum(p.numel() for p in head.parameters())/n:.4%} của model)\n")

t0 = time.time(); tot = None
for ep in range(EPOCH):
    random.shuffle(idx); s = 0.0; nb = 0
    head.train()
    for j in range(0, len(idx), BS):
        lo = idx[j:j+BS]
        v = Vtr_g[lo]                                  # [B, 8, 3, H]
        # chấm TỪNG đoạn rồi MAX trên trục đoạn -> điểm văn bản. Phải khớp `cham()`
        # ở Bước 4, nếu không thì huấn luyện một đằng, đánh giá một nẻo.
        lg = cham_vec(head, v.reshape(-1, v.shape[-1])).view(len(lo), K, K_CHUNK).max(-1).values
        # softmax listwise: chỉ đòi gold CAO NHẤT NHÓM. Negative liên quan
        # KHÔNG bị ép về 0 -> đúng chỗ BCE nhãn cứng đã dạy sai hai lần trước.
        loss = F.cross_entropy(lg, tgt[lo])
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        s += loss.item(); nb += 1
    if ep % 5 == 0 or ep == EPOCH-1:
        r, _ = cham(head, Vdv)
        print(f"  epoch {ep+1:>2}  loss {s/nb:.4f}  dev R@5 {r[5]:.4f} ({(r[5]-MOC[5])*100:+.2f})", flush=True)
print(f"\n{(time.time()-t0)/60:.1f} phút")



In [ ]:
# ===== Bước 6: kết luận =====
MOI, per = cham(head, Vdv)
d5, d1 = MOI[5]-MOC[5], MOI[1]-MOC[1]
print(f"{'':12}{'R@1':>9}{'R@5':>9}")
print(f"{'đầu gốc':<12}{MOC[1]:>9.4f}{MOC[5]:>9.4f}")
print(f"{'đầu học lại':<12}{MOI[1]:>9.4f}{MOI[5]:>9.4f}")
print(f"{'Δ':<12}{d1:>+9.4f}{d5:>+9.4f}")

dat = d5 >= NGUONG and MOI[5] > SAN and HOP_LE
print("\n" + "="*70)
if dat:
    print(f"✅ THÔNG TIN NẰM SẴN TRONG BIỂU DIỄN, chỉ bị chấm lệch (và vượt SÀN {SAN:.4f}).")
    print("   -> LoRA rất đáng chạy: nó có nhiều sức hơn đầu tuyến tính rất nhiều.")
else:
    print("❌ Đổi cách chấm KHÔNG cứu được gì.")
    print("   -> Biểu diễn không chứa tín hiệu. Tường ngữ nghĩa là tường THẬT.")
    print("   -> ĐỪNG chạy LoRA 3h. Đóng hẳn hướng fine-tune.")
print("="*70)

json.dump({"moc": MOC, "moi": MOI, "delta_r5": d5, "dat": bool(dat),
           "san": SAN, "hop_le": bool(HOP_LE), "k_chunk": K_CHUNK, "n_nhom": len(nhom), "K": K, "bo_top": BO_TOP, "lr": LR, "epoch": EPOCH},
          open(f"{OUT}/meta_probe.json", "w", encoding="utf-8"))
json.dump(per,     open(f"{OUT}/scores_dev300_probe_k3.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump(PER_MOC, open(f"{OUT}/scores_dev300_moc_k3.json",   "w", encoding="utf-8"), ensure_ascii=False)
print("\nTẢI outputs/ VỀ TRƯỚC KHI ĐÓNG PHIÊN.")
